In [74]:
#
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser



"""
Task: ask a question and get an answer from the LLM.
"""

'\nTask: ask a question and get an answer from the LLM.\n'

In [75]:
load_dotenv()  # Load environment variables from .env file

True

In [76]:
# initialize the model
model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, max_tokens=300)
strOutputParser = StrOutputParser()

In [51]:
model.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E8LWRK5EuzWSjJJ1TuBZDaR58qGxS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fc183-b906-7b52-9e83-02abd66dcc47-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [63]:
class QNA(TypedDict):
    question: str
    answer: str

In [66]:
def get_answer(input: QNA) -> QNA:
    question = input["question"]
    chain = model | strOutputParser
    answer = chain.invoke(question)
    return {"question": question, "answer": answer}


In [67]:
#initialize the langgraph.
state_graph = StateGraph(QNA)

In [68]:
state_graph.add_node("get_answer", get_answer)
state_graph.add_edge(START, "get_answer")
state_graph.add_edge("get_answer", END)

In [69]:
workflow = state_graph.compile()

In [70]:
initialState = {"question": "What is the capital of France?"}
result = workflow.invoke(initialState)
print(result)  # Output: {'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}


{'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}
